In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

## Chat Models: GPT-3.5 Turbo and GPT-4

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()
output = llm.invoke('Explain quantun mechanincs in one sentence.')
print(output.content)

Quantum mechanics is the branch of physics that describes the behavior of particles at a very small scale, where interactions are governed by probabilities rather than definite outcomes.


In [5]:
help(ChatOpenAI)

Help on class ChatOpenAI in module langchain_openai.chat_models.base:

class ChatOpenAI(BaseChatOpenAI)
 |  ChatOpenAI(
 |      *args: Any,
 |      name: str | None = None,
 |      cache: langchain_core.caches.BaseCache | bool | None = None,
 |      verbose: bool = <factory>,
 |      callbacks: list[langchain_core.callbacks.base.BaseCallbackHandler] | langchain_core.callbacks.base.BaseCallbackManager | None = None,
 |      tags: list[str] | None = None,
 |      metadata: dict[str, typing.Any] | None = None,
 |      custom_get_token_ids: collections.abc.Callable[[str], list[int]] | None = None,
 |      rate_limiter: langchain_core.rate_limiters.BaseRateLimiter | None = None,
 |      disable_streaming: Union[bool, Literal['tool_calling']] = False,
 |      output_version: str | None = <factory>,
 |      profile: langchain_core.language_models.model_profile.ModelProfile | None = None,
 |      client: Any = None,
 |      async_client: Any = None,
 |      root_client: Any = None,
 |      roo

If you check OpenAI documentation, there is something called a list of messages being sent to the LLM.  
Each message has a role assigned to it  
System -> Sets the behavior of the LLM  
user -> Sets the actual prompt of the user  
assistant -> Store prior responses.  


In [35]:

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

messages = [
    SystemMessage(content="You are a physicist and respond only in German"),
    HumanMessage(content="Exaplain quantum mechanics in one sentence")
]

output = llm.invoke(messages)
print(output.content)



Quantenmechanik ist ein Teilgebiet der Physik, das das Verhalten von Materie und Energie auf der kleinsten Skala von Atomen und subatomaren Teilchen beschreibt, wobei Phänomene wie Wellen-Teilchen-Dualität, Unschärferelation und Verschränkung zentrale Rollen spielen.


### Caching
Practice of storing frequently accessed data or result in a temporary, faster storage layer.  
it optimizes the interaction with LLMs by reducing API calls and speeding up allications, resulting in a more efficient user experience.

#### In-Memory Cache

In [7]:
from langchain_core.globals import set_llm_cache
from langchain_openai import OpenAI
llm = OpenAI(model_name='gpt-3.5-turbo-instruct')

In [8]:
%%time
from langchain_core.caches import InMemoryCache
set_llm_cache(InMemoryCache())
prompt = 'Tell me a joke that a toddler can understand'
llm.invoke(prompt)

CPU times: total: 0 ns
Wall time: 1.85 s


'\n\nWhy did the tomato turn red?\n\nBecause it saw the salad dressing!'

In [9]:
%%time
llm.invoke(prompt)

CPU times: total: 0 ns
Wall time: 228 μs


'\n\nWhy did the tomato turn red?\n\nBecause it saw the salad dressing!'

Take a look at the time difference. Since the prompt response is already in the cache the reponse is much faster.

#### SQLite Caching

In [10]:
from langchain_community.cache import SQLiteCache
set_llm_cache(SQLiteCache(database_path='.langchain.db'))

# First request (not in cache, takes longer)
llm.invoke("Tell me a joke")

# Second request (cache, faster)
llm.invoke("Tell me a joke")

'\n\nWhy was the math book sad?\n\nBecause it had too many problems. '

### LLM Streaming

In [11]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()
prompt = 'Write a rock song about the Moon and a Raven'
print(llm.invoke(prompt).content)

(Verse 1)
In the still of the night, under the pale moonlight
A raven takes flight, soaring through the night
Its wings beat against the sky, a shadow in the dark
As the moon watches on, a silent and mysterious spark

(Chorus)
Oh moon and raven, dancing in the night
A haunting and beautiful sight
They share a connection, ancient and deep
A bond that mere mortals can't keep

(Verse 2)
The raven calls out, a lonely cry
Echoing through the night, reaching for the sky
The moon listens, a silent sentinel above
Watching over the world with a gentle love

(Chorus)
Oh moon and raven, dancing in the night
A haunting and beautiful sight
They share a connection, ancient and deep
A bond that mere mortals can't keep

(Bridge)
The moon shines bright, casting its silver glow
The raven flies free, with a wisdom we'll never know
Together they move in harmony, a cosmic dance
A celestial romance, forever in trance

(Chorus)
Oh moon and raven, dancing in the night
A haunting and beautiful sight
They share

In [12]:
### LLM Streaming
for chunk in llm.stream(prompt):
    print(chunk.content, end='', flush=True)

Verse 1:
Underneath the silver glow, 
In the dead of night we roam,
The raven calls out to the Moon,
Singing a haunting tune,
On wings of darkness we take flight,
Guided by the pale moonlight,
Together we're a force to behold,
In the sky, so dark and cold,

Chorus:
Moon and Raven, allies in the night,
Together we'll conquer, we'll take flight,
In the darkness, we'll reign supreme,
Moon and Raven, a rock n' roll dream,

Verse 2:
The Moon, a silent watcher above,
Shining down with a mystical love,
The raven, a messenger of fate,
Guiding us through the night's embrace,
We soar through the midnight sky,
Together we'll never die,
An unbreakable bond, forged in the dark,
Moon and Raven, leaving our mark,

Chorus:
Moon and Raven, allies in the night,
Together we'll conquer, we'll take flight,
In the darkness, we'll reign supreme,
Moon and Raven, a rock n' roll dream,

Bridge:
Soaring high, on wings of black,
With the Moon by our side, we'll never look back,
Together we'll conquer, we'll never

### Prompt Templates 

Langchain has templates which can be used in setup of prompts.  
PromptTemplate are a way to create dynamic prompts for LLms

In [14]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

template = '''You are an experienced virologist.
Write a few sentence about the following virus "{virus}" in {language}'''

prompt_template = PromptTemplate(template=template)
prompt = prompt_template.format(virus='covid19', language='german')
prompt 

'You are an experienced virologist.\nWrite a few sentence about the following virus "covid19" in german'

In [17]:
llm = ChatOpenAI(model_name='gpt-3.5-turbo', temperature=0)
output = llm.invoke(prompt)
print(output.content)

Das Covid-19-Virus, auch bekannt als das Coronavirus, ist ein neuartiges Virus, das erstmals in Wuhan, China, identifiziert wurde. Es gehört zur Familie der Coronaviren und verursacht eine Atemwegserkrankung, die von milden Symptomen bis hin zu schweren Lungenentzündungen reichen kann. Die Verbreitung des Virus hat zu einer weltweiten Pandemie geführt und erfordert umfangreiche Maßnahmen zur Eindämmung und Bekämpfung.


### ChatPromptTemplates
Specifically used for chats

In [20]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content='You respond only in JSON format'),
        HumanMessagePromptTemplate.from_template('Top {n} countries in {area} by population.')
    ]
)
messages = chat_template.format_messages(n='10', area='europe')
print(messages)

[SystemMessage(content='You respond only in JSON format', additional_kwargs={}, response_metadata={}), HumanMessage(content='Top 10 countries in europe by population.', additional_kwargs={}, response_metadata={})]


In [21]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI()
output = llm.invoke(messages)
print(output.content)

{
    "countries": [
        {
            "name": "Germany",
            "population": 83783942
        },
        {
            "name": "United Kingdom",
            "population": 67886011
        },
        {
            "name": "France",
            "population": 65273511
        },
        {
            "name": "Italy",
            "population": 60461826
        },
        {
            "name": "Spain",
            "population": 46754778
        },
        {
            "name": "Ukraine",
            "population": 43733762
        },
        {
            "name": "Poland",
            "population": 37887768
        },
        {
            "name": "Romania",
            "population": 19705301
        },
        {
            "name": "Netherlands",
            "population": 17134872
        },
        {
            "name": "Belgium",
            "population": 11589623
        }
    ]
}


## Chains

In [22]:
# Update all the models first
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()
chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You are a visionary scientist from the future."),
        HumanMessagePromptTemplate.from_template(
            'Describe how humans will explore "{location}" in the year {year},'
            'using advanced technologies'
        )
    ]
)
# This is how chaining works
chain = chat_template | llm
output = chain.invoke({'location': 'The land of India', 'year': '2150'})
print(output.content)


In the year 2150, humans will explore the land of India using a combination of advanced technologies that have revolutionized the way we discover and study new environments. One of the key technologies that will play a crucial role in this exploration is advanced virtual reality (VR) and augmented reality (AR) systems. These systems will allow researchers and explorers to immerse themselves in realistic and detailed virtual environments of India without physically being there. This will help minimize the impact on the environment while still allowing for in-depth exploration.

Furthermore, drones equipped with advanced sensors and cameras will be used extensively to map out the diverse landscapes of India, from the lush forests of the Western Ghats to the arid deserts of Rajasthan. These drones will be capable of capturing high-resolution images and data that can be analyzed in real-time to study and monitor various aspects of the Indian terrain, such as biodiversity, geology, and clim

In [23]:
output.response_metadata

{'token_usage': {'completion_tokens': 411,
  'prompt_tokens': 41,
  'total_tokens': 452,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_provider': 'openai',
 'model_name': 'gpt-3.5-turbo-0125',
 'system_fingerprint': None,
 'id': 'chatcmpl-DHp6mGEbhmzeaR2OGGt0VoEDyAOJ7',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}

## Google Gemini Integration with Langchain

In [24]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [25]:
import getpass
import os
if 'GOOGLE_API_KEY' not in os.environ:
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Provide yoru Google API Key')

In [26]:
import google.generativeai as genai
for model in genai.list_models():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/gemini-embedding-2-

C:\Users\ashut\AppData\Local\Temp\ipykernel_15276\1397781936.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [29]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.9)
response = llm.invoke('Write a pragraph about life on Mars in year 2100')
print(response.content)

By 2100, Mars is no longer a distant dream but a burgeoning frontier, home to tens of thousands of pioneering settlers. Life thrives within vast, interconnected subterranean complexes and gleaming surface domes, shielded from the harsh radiation and thin atmosphere. Daily routines revolve around resource management – recycling every drop of water, optimizing hydroponic farms for sustenance, and meticulously maintaining the complex life support systems that breathe artificial air into their world. Children play in simulated parks under UV-filtered skylights, while engineers and scientists tirelessly work on early-stage terraforming projects and excavate rare minerals from the rust-red soil. Though the stark, alien landscape outside remains a constant reminder of their isolation, a unique Martian culture is taking root, defined by resilience, innovation, and a shared dream of transforming a barren world into a second cradle for humanity.


In [31]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

prompt = PromptTemplate.from_template('You are a content creator. Write me a tweet about {topic}')
chain = LLMChain(
    llm=llm,
    prompt=prompt,
    verbose=True
)

topic = "Why will AI change the world"
response = chain.invoke(input=topic)
print(response)




> Entering new LLMChain chain...
Prompt after formatting:
You are a content creator. Write me a tweet about Why will AI change the world

> Finished chain.
{'topic': 'Why will AI change the world', 'text': "Here are a few options, playing with slightly different angles and vibes, perfect for a content creator!\n\n---\n\n**Option 1 (Enthusiastic & Broad Impact):**\n\n> Why will AI change the world? 🤯 Because it's not just a tool, it's a *fundamental shift* in how we live, work, and create! From revolutionizing medicine to sparking new forms of art, it's unlocking human potential and solving problems at a scale we've never seen. The future is now! ✨ #AI #FutureIsNow #Innovation\n\n---\n\n**Option 2 (Intriguing & Focus on Augmentation):**\n\n> Forget incremental updates. AI is ushering in an era where human ingenuity is amplified like never before. 🧠 It's not just automating tasks; it's *augmenting* our ability to discover, personalize, and tackle the world's biggest challenges. Get rea

### System Prompt and Streaming

In [34]:
from langchain_core.messages import HumanMessage, SystemMessage
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
output = llm.invoke(
    [
        SystemMessage(content="Answer only YES or NO in French."),
        HumanMessage(content='Is fish a mammal?')
    ]
)
output.content

'NON'

### Streaming

In [35]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)
prompt = 'Write a scientific paper outlining the mathematical foundation of our universe.'

for chunk in llm.stream(prompt):
    print(chunk.content)


## The Intrinsic Symphony: A Mathematical Foundation of the Universe


**Authors:** Dr. A. K. Sharma, Dr. L. M. Chen, Prof. S. R. Gupta
**Affiliation:** Institute for Theoretical Cosmology and Fundamental Physics, Global University Consortium

---

### Abstract


The universe, in its grandest and most microscopic manifestations, appears to be fundamentally mathematical. This paper explores the profound role of mathematics not merely as a descriptive tool, but as the intrinsic language and underlying structure of reality. We delineate the key mathematical frameworks
 that underpin our understanding of the cosmos, from the differential geometry governing spacetime and gravity to the linear algebra and complex analysis dictating quantum mechanics, and the group theory elucidating fundamental symmetries and forces. We discuss the challenges in unifying these frameworks and
 the mathematical innovations required for a complete theory of everything, touching upon concepts like higher-dimensi

### Miltimodal AI with Gemini

Get the image and convert it to base64.  
Create a humanmessage template  
Write a prompt  
Use template and the prompt to invoke the llm to describe what is seen in the image.

In [1]:
from IPython.display import Image
import base64

def encode_image(image_path):
    with open(image_path, 'rb') as f:
        image_data = f.read()
        encoded_image_bytes = base64.b64encode(image_data)
        base64_string = encoded_image_bytes.decode('utf-8')
        return base64_string

byte_image = encode_image('match.jpg')

In [2]:
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)
prompt = 'What is in this image?'

message = HumanMessage(
    content=[
        {
            'type': 'text',
            'text': prompt
        },
        {
            'type': 'image_url',
            'image_url': { 'url': f'data:image/jpeg;base64, {byte_image}'}
        }
    ]
)
response = llm.invoke([message])

In [4]:
print(response.content)

This image captures an outdoor sports stadium during what appears to be a **rugby match**.

Here's a detailed breakdown of what's in the image:

*   **Foreground:** The bottom of the image shows a dark barrier or railing, suggesting the photo was taken from behind it, likely from the stands. On the green grass field, a blue plastic bottle is visible lying on its side.
*   **The Field:** A large, well-maintained green grass field dominates the lower and middle sections of the image. White lines are clearly marked on the grass, indicating the boundaries and specific areas of play for a rugby pitch.
*   **Players:** Numerous players are on the field. Two teams are distinguishable by their uniforms: one team wears dark (possibly maroon or brown) jerseys and shorts, while the other wears white jerseys and dark shorts. Many players are grouped together in formations typical of rugby, such as a scrum or ruck. Some players are standing individually.
*   **Officials/Equipment:** Yellow flags or

In [5]:
def ask_gemini(prompt, image_path, model="gemini-2.5-flash"):
    byte_image = encode_image(image_path)
    llm = ChatGoogleGenerativeAI(model=model, temperature=0.2)
    messages = HumanMessage(
        content=[
            {
                'type': 'text',
                'text': prompt
            },
            {
                'type': 'image_url',
                'image_url': { 'url': f'data:image/jpeg;base64, {byte_image}' }
            }
        ]
    )
    response = llm.invoke([messages])
    return response.content

In [6]:
print(ask_gemini("What kind of sport is this?", "match.jpg"))

Based on the image, this is a game of **Rugby**.

Here's why:
1.  **H-shaped Goalposts:** You can clearly see the distinctive H-shaped goalposts at the end of the field (especially in crops 3 and 4). These are characteristic of rugby.
2.  **Player Formation:** In the middle of the field, there's a cluster of players in a tight, physical formation (visible in crops 3, 4, 5, and 6). This is either a **scrum** or a **ruck**, which are fundamental elements of rugby.
3.  **Field Markings:** The field has specific white lines, including what appear to be 22-meter lines and a halfway line, typical of a rugby pitch.
4.  **Player Attire:** Players are wearing jerseys and shorts, consistent with rugby attire.


### Langchain Tools: DuckDuckGo and Wikipedia

#### DuckDuckGo

In [9]:
from langchain_community.tools import DuckDuckGoSearchRun

In [11]:
search = DuckDuckGoSearchRun()
output = search.invoke("Where was Freddie Mercury born?")
print(output)

Born in 1946 inZanzibarto Indian parents from the Parsi community, Mercury attended British boarding schools in India from the age of eight and returned to Zanzibar after secondary school. In 1964, his family fled the Zanzibar Revolution, moving to Middlesex, England. Feb 18, 2026 ·Born Farrokh Bulsara in 1946 to Parsi parents inZanzibar, Freddie Mercury attended boarding school in Panchgani, Maharashtra, India, where he took piano lessons. Jan 15, 2025 ·Freddie Mercury, born Farrokh Bulsara on September 5, 1946, inZanzibar, was the flamboyant lead singer of the legendary rock band Queen. Renowned for his powerful vocals and dynamic stage presence, Mercury is recognized as one of the most talented and innovative musicians of the rock era. Freddie Mercury was born on theTanzanian island of Zanzibar. His parents, Bomi and Jer Bulsara, sent him off to a private school in India, from 1955 til 1963. In 1964, he and his family flew to England. In 1966 he started his education at the Ealing C

In [12]:
from langchain_community.tools import DuckDuckGoSearchResults
search = DuckDuckGoSearchResults()
output = search.run("Freddie Mercury and Queen")
print(output)

snippet: FreddieMercury(born Farrokh Bulsara ; 5 September 1946 – 24 November 1991) 2 was a British singerandsongwriter who achieved worldwide fame as ..., title: Freddie Mercury - Wikipedia, link: https://en.wikipedia.org/wiki/Freddie_Mercury, snippet: FreddieMercuryandQueen. ...FreddieMercury&Queenfan walking tour , who made a (thankfully sunny in London) Sunday afternoon all the more fun ..., title: 2026 Freddie Mercury and Queen. The London tour - with Trusted, link: https://www.tripadvisor.com/AttractionProductReview-g186338-d17823500-Freddie_Mercury_and_Queen_The_London_tour-London_England.html, snippet: The OfficialFreddieMercuryBirthday Party 2026: Merchandise Range ...FreddieMercuryOfficial Website www.freddiemercury.com, title: QueenOnline.com - The Official Queen Website, link: https://www.queenonline.com/, snippet: Queen:FreddieMercuryandBrian ...QueensFreddieMercuryandBrian May jamming in 1986 is a rare glimpse of the rock stars behind closed doors., title: Queen: Freddie 

In [13]:
# Wrapper for duck duck go
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

wrapper = DuckDuckGoSearchAPIWrapper(region='de-de', max_results=3, safesearch='moderate')
search = DuckDuckGoSearchResults(api_wrapper=wrapper, source='news')
output = search.run('Berlin')
print(output)

snippet: Berlin ([bɛr'li:n] ) ist die Hauptstadt und ein Land der Bundesrepublik Deutschland. Die Großstadt ist mit rund 3,7 Millionen Einwohnern die bevölkerungsreichste und mit 891 Quadratkilometern die flächengrößte Gemeinde Deutschlands sowie die bevölkerungsreichste Stadt der Europäischen Union. In der Agglomeration Berlin leben mehr als 4,7 Millionen Menschen und in der Metropolregion Berlin-Brandenburg mehr als 6,2 Millionen. Der Stadtstaat besteht aus zwölf Bezirken. Neben den Flüssen Spree, Havel und Dahme befinden sich im Stadtgebiet kleinere Fließgewässer sowie zahlreiche Seen und Wälder.Berlin wurde im 13. Jahrhundert erstmals urkundlich erwähnt. Die Stadt war Hauptstadt der Mark Brandenburg, Preußens und Deutschlands. Im Laufe des 19. und frühen 20. Jahrhunderts entwickelte sich Berlin zur weltweit drittgrößten Stadt. Nach dem Ende des Zweiten Weltkriegs unterlag die Stadt 1945 dem Viermächtestatus; Ost-Berlin war ab 1949 de facto Hauptstadt der Deutschen Demokratischen Re

In [36]:
import re

pattern = r'snippet: (.*?), title: (.*?), link: (,*?)\],'
matches = re.findall(pattern, output, re.DOTALL)

for snippet, title, link in matches:
    print(f'Snippet: {snippet}\nTitle: {title}\nLink: {link}\n')
    print('-' * 50)

TypeError: expected string or bytes-like object, got 'AIMessage'

#### Wikipedia tool

In [16]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [17]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)
wiki.invoke({'query': 'llamaindex'})

'Page: Vector database\nSummary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically implement approximate nearest neighbor algorithms so users can search for records semantically similar to a given input, unlike traditional databases which primarily look up records by exact match. Use-cases for vector databases include similarity search, semantic search, multi-modal search, recommendations e'

In [18]:
wiki.invoke("Google Gemini")

'Page: Google Gemini\nSummary: Gemini (also known as Google Gemini and formerly known as Bard) is a generative artificial intelligence chatbot and virtual assistant developed by Google. It is powered by the large language model (LLM) of the same name, after previously being based on LaMDA and PaLM 2.\nThe Gemini architecture is trained natively on multiple data types, allowing the models to process and generate text, computer code, images, audio, and video simultaneously. Google distributes the tec'

### Create a ReAct Agent

In [19]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic import hub
from langchain_classic.agents import Tool, AgentExecutor, initialize_agent, create_react_agent
from langchain_classic.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_openai import ChatOpenAI



In [ ]:
# Create the llm
llm = ChatOpenAI(model="gpt-4-turbo-preview", temperature=0)

In [ ]:
#Create a prompt template
template = '''
Answer the following questions as best you can.
Question: {q}
'''
prompt_template = PromptTemplate.from_template(template)

# leveraging React prompt from the hub
prompt = hub.pull('hwchase17/react')
# print(type(prompt))
# print(prompt.input_variables)
# print(prompt.template)



<class 'langchain_core.prompts.prompt.PromptTemplate'>
['agent_scratchpad', 'input', 'tool_names', 'tools']
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [29]:
# We will use 3 tools so the REact agent will decide which tool to use depending on the question asked
# This is the basics of react agents

# 1. Python REPL Tool (for executing Python code)
python_repl = PythonREPLTool()
python_repl_tool= Tool(
    name="Python REPL",
    func=python_repl.run,
    description='Useful when you need to use Python to answer a question. You should input Python code.'
)

# 2. Wikipedia Tool (for searching Wikipedia)

api_wrapper = WikipediaAPIWrapper()
wikipedia = WikipediaQueryRun(api_wrapper=api_wrapper)
wikipedia_tool = Tool(
    name="Wikipedia",
    func=wikipedia.run,
    description="Useful for when you need to look up a topic, country, or person on Wikipedia."
)

# 3. DuckDuckGo Search Tool (for general web searches)
search = DuckDuckGoSearchRun()
duckduckgo_tool = Tool(
    name="DuckDuckGo Search",
    func=search.run,
    description="Useful for when you need to perform an internet search to find information that another tool can't provide"
)

tools = [python_repl_tool, wikipedia_tool, duckduckgo_tool]

# This line combines the llm , tools and the prompt and make it reactive
agent = create_react_agent(llm, tools, prompt)

# Agent Executor manages and executes the agents output
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True, # helps the executor what to do when parsing errors occurs
    max_iterations=10  # Max number of iterations the agent should be executed.
)

In [30]:
# Question 1
question = 'Generate the first 20 numbers in the Fibonacci series.'
output = agent_executor.invoke({
    'input': prompt_template.format(q=question)
})



> Entering new AgentExecutor chain...


Python REPL can execute arbitrary code. Use with caution.


I can generate the first 20 numbers in the Fibonacci series using a simple Python script.

Action: Python REPL

Action Input:
```python
def fibonacci(n):
    fib_series = [0, 1]
    for i in range(2, n):
        next_fib = fib_series[-1] + fib_series[-2]
        fib_series.append(next_fib)
    return fib_series[:n]

fibonacci(20)
```
I now know the first 20 numbers in the Fibonacci series.

Final Answer: The first 20 numbers in the Fibonacci series are [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181].

> Finished chain.


In [31]:
print(output['input'])


Answer the following questions as best you can.
Question: Generate the first 20 numbers in the Fibonacci series.



In [32]:
print(output['output'])

The first 20 numbers in the Fibonacci series are [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181].


In [33]:
# Question 2
question = 'Who is the current prime minister of the UK.'
output = agent_executor.invoke({
    'input': prompt_template.format(q=question)
})



> Entering new AgentExecutor chain...
I need to find the current Prime Minister of the United Kingdom as of my last update.

Action: Wikipedia

Action Input: Prime Minister of the United Kingdom
Page: Prime Minister of the United Kingdom
Summary: The prime minister of the United Kingdom is the head of government of the United Kingdom. The prime minister advises the sovereign on the exercise of much of the royal prerogative, chairs the Cabinet, and selects its ministers. Modern prime ministers hold office by virtue of their ability to command the confidence of the House of Commons, so they are invariably members of Parliament.
The office of prime minister is not established by any statute or constitutional document, but exists only by long-established convention, whereby the monarch appoints as prime minister the person most likely to command the confidence of the House of Commons. In practice, this is the leader of the political party that holds the largest number of seats in the Com

In [34]:
# Question 3
question = "Tell me about Napolean Bonaparte early life."
output = agent_executor.invoke({
    'input': prompt_template.format(q=question)
})



> Entering new AgentExecutor chain...
I should use Wikipedia to find detailed information about Napoleon Bonaparte's early life.
Action: Wikipedia
Action Input: Napoleon Bonaparte early lifePage: Napoleon III
Summary: Napoleon III (born Charles-Louis Napoléon Bonaparte; 20 April 1808 – 9 January 1873) was President of France from 1848 to 1852 and then Emperor of the French from 1852 until his deposition in 1870. He was the first president, second emperor, and last monarch of France. He created the Second French Empire in 1852 and this period saw rapid industrialization in France, rapid expansion of infrastructure and rise of French influence in world politics after several decades of instability. He was the son of Louis Bonaparte, King of Holland and the nephew of Napoleon, Emperor of the French. As head of state of France for 22 years, he was the longest-reigning French head of state since the end of the  ancien régime.        
Napoleon III was born at the height of the First French